# 02 — Rebuild training-only FAISS galleries from pre-encoded vectors

Pre-encoded train vectors are aligned to the clean gallery by identifiers. Pre-encoded test vectors become query matrices only; test indexes are never searched. Positional matching is disabled. Every saved neighbor is checked again against all leakage rules.

In [ ]:
from pathlib import Path
import json, os, sys

def find_rerun_dir():
    candidates = [
        Path(os.environ.get("JAMIA_RERUN_DIR", "")),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if str(candidate) and (candidate / "rerun_config.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Set JAMIA_RERUN_DIR to the folder containing rerun_config.json")

RERUN_DIR = find_rerun_dir()
# Never expose the implementation directory as a top-level import location:
# rerun_code/statistics.py would shadow Python's standard-library statistics.
implementation_dir = (RERUN_DIR / "src" / "rerun_code").resolve()
clean_sys_path = []
for entry in sys.path:
    try:
        resolved_entry = Path(entry or ".").resolve()
    except Exception:
        resolved_entry = None
    if resolved_entry != implementation_dir:
        clean_sys_path.append(entry)
sys.path[:] = clean_sys_path
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config, output_paths
RERUN_DIR, CONFIG = load_config(RERUN_DIR)
PATHS = output_paths(CONFIG)
print("Code:", RERUN_DIR)
print("Output:", PATHS["root"])

In [ ]:
import numpy as np, pandas as pd
from rerun_code.common import write_json
from rerun_code.preencoded import load_preencoded, align_manifest_to_preencoded, inventory_preencoded
from rerun_code.leakage_safe_data import read_jsonl, assert_leakage_free
from rerun_code.leakage_safe_retrieval import save_gallery_bundle, save_query_bundle, load_gallery_bundle, load_query_bundle, safe_search, assert_neighbor_records, write_metadata

built, provenance = {}, {}
for dataset in ("mimic", "iuhn"):
    gallery = read_jsonl(PATHS["manifests"] / dataset / "clean_gallery.jsonl")
    queries = read_jsonl(PATHS["manifests"] / dataset / "fixed_queries.jsonl")
    assert_leakage_free(gallery, queries, phash_threshold=CONFIG["phash_threshold"])
    spec = CONFIG["datasets"][dataset]
    train_vectors, train_meta, train_prov = load_preencoded(spec["preencoded_train"])
    test_vectors, test_meta, test_prov = load_preencoded(spec["preencoded_test"])
    print(dataset, "training image vectors:", train_prov["vector_file"])
    print(dataset, "test/query image vectors:", test_prov["vector_file"])
    gallery_vectors, _, gallery_alignment = align_manifest_to_preencoded(gallery, train_vectors, train_meta, allow_positional=False)
    query_vectors, _, query_alignment = align_manifest_to_preencoded(queries, test_vectors, test_meta, allow_positional=False)
    root = PATHS["bundles"] / dataset
    save_gallery_bundle(gallery_vectors, gallery.to_dict("records"), root / "gallery")
    save_query_bundle(query_vectors, queries.to_dict("records"), root / "queries")
    built[dataset] = (gallery_vectors, gallery, query_vectors, queries)
    provenance[dataset] = {"train": train_prov, "test": test_prov, "gallery_alignment": gallery_alignment, "query_alignment": query_alignment}

In [ ]:
combined_gallery_vectors = np.concatenate([built[d][0] for d in ("mimic", "iuhn")], axis=0)
combined_gallery = pd.concat([built[d][1] for d in ("mimic", "iuhn")], ignore_index=True)
combined_query_vectors = np.concatenate([built[d][2] for d in ("mimic", "iuhn")], axis=0)
combined_queries = pd.concat([built[d][3] for d in ("mimic", "iuhn")], ignore_index=True)
expected_gallery = read_jsonl(PATHS["manifests"] / "combined" / "clean_gallery.jsonl")
expected_ids = expected_gallery["record_id"].tolist()
index_by_id = {value: i for i, value in enumerate(combined_gallery["record_id"].tolist())}
positions = [index_by_id[value] for value in expected_ids]
combined_gallery_vectors = combined_gallery_vectors[positions]
combined_gallery = combined_gallery.iloc[positions].reset_index(drop=True)
root = PATHS["bundles"] / "combined"
save_gallery_bundle(combined_gallery_vectors, combined_gallery.to_dict("records"), root / "gallery")
save_query_bundle(combined_query_vectors, combined_queries.to_dict("records"), root / "queries")
provenance["combined_legacy_inventory_only"] = {
    "train": inventory_preencoded(CONFIG["datasets"]["combined"]["preencoded_train"]),
    "test": inventory_preencoded(CONFIG["datasets"]["combined"]["preencoded_test"]),
    "note": "Primary combined bundle was assembled from the two identifier-aligned, cleaned component bundles."
}

In [ ]:
for dataset in ("mimic", "iuhn", "combined"):
    root = PATHS["bundles"] / dataset
    index, gallery_metadata = load_gallery_bundle(root / "gallery")
    query_vectors, query_metadata = load_query_bundle(root / "queries")
    neighbors = []
    for vector, query in zip(query_vectors, query_metadata):
        found = safe_search(index, gallery_metadata, vector, query, k=CONFIG["retrieval_k"], phash_threshold=CONFIG["phash_threshold"])
        assert_neighbor_records(query, found, expected_k=CONFIG["retrieval_k"], phash_threshold=CONFIG["phash_threshold"])
        neighbors.append({"query_record_id": query["record_id"], "neighbors": found})
    write_metadata(neighbors, root / "neighbors.jsonl")
    exact_like = [{"dataset": dataset, "query": row["query_record_id"], "rank1": row["neighbors"][0]["cosine_score"]} for row in neighbors if row["neighbors"][0]["cosine_score"] >= 0.9999]
    write_json(root / "rank1_score_ge_0_9999.json", exact_like)
    if exact_like: print("MANUAL REVIEW REQUIRED:", dataset, len(exact_like), "rank-1 scores >= 0.9999")
write_json(PATHS["bundles"] / "preencoded_provenance.json", provenance)
print("BUNDLE GATE PASSED. Review every rank-1 score >= 0.9999 before generation.")